# Preparation 1 — Build the frozen final-test bundle

This CPU-only notebook packages the already-frozen Phase 3.5B test split (683 clips). It extracts original WAV bytes from the source Dataset A Parquet shards, checks each decoded waveform against the frozen manifest, applies no audio transformations, and creates a **private** ZIP. It does not train, tune, run inference, or change split membership.

Run this notebook locally from the BTP project with the BTP Python environment. The original shard files must be present under `raw/A_narration/data/`. It uses the reviewed preparation script in `scripts/prepare_phase7_test_bundle.py`.

The test references remain sealed: use this bundle only after model and decoding choices are frozen. Never attach it to training or tuning runs.


## 1. Locate BTP and check whether the bundle already exists

The preparation script refuses to overwrite an existing bundle. If the verified bundle is present, this cell reports it and the following verification cell checks its archive.

In [ ]:
# Locate the repository from this notebook, then define the preparation artifacts.
from pathlib import Path
import json, shutil, sys, runpy, zipfile, hashlib
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "scripts/prepare_phase7_test_bundle.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter in the BTP project or one of its subfolders.")
SCRIPT = ROOT / "scripts/prepare_phase7_test_bundle.py"
OUT = ROOT / "outputs/phase7_final_test_bundle_private_v1"
ZIP = OUT / "phase7_final_test_bundle_private.zip"
assert (ROOT / "raw/A_narration/data").is_dir(), "Original Dataset A Parquet shards are missing."
print({"project": str(ROOT), "source_shards_available": True, "bundle_exists": ZIP.is_file()})
# Build once if absent; otherwise keep the verified existing version.
if not ZIP.is_file():
    runpy.run_path(str(SCRIPT), run_name="__main__")
else:
    print("Bundle already exists; proceeding to verification without rebuilding.")


## 2. Verify the private ZIP and its contents

This checks the saved bundle summary, ZIP CRC integrity, the presence of exactly 683 WAV clips, and that the split marker is test. The bundle and its references must remain private.

In [ ]:
# Read the preparation summary and verify the ZIP archive contents.
summary = json.loads((OUT / "test_bundle_summary_private.json").read_text())
record = json.loads((OUT / "phase7_final_test_bundle/bundle_record.json").read_text())
assert summary["status"] == "passed" and summary["clips"] == 683
assert summary["train_or_validation_rows_included"] is False
assert record["evaluation_partition"] == "test" and record["no_training_or_tuning"] is True
with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None
    members = archive.namelist()
    wavs = [name for name in members if name.startswith("audio/") and name.endswith(".wav")]
    assert len(wavs) == 683, f"Expected 683 test WAVs, found {len(wavs)}"
print({"status": "VERIFIED", "clips": len(wavs), "hours": round(summary["hours"], 6),
       "zip_size_mib": round(ZIP.stat().st_size / 1024**2, 1),
       "sha256": summary["zip_sha256"], "path": str(ZIP)})
